In [0]:
%sql
use catalog workspace;
use schema default;

#### Creating a Streaming Dataframe

In [0]:
from pyspark.sql.functions import col, current_timestamp, expr
stream_df = spark.readStream.format("rate").option("rowsPerSecond", 1).load()\
    .select(
        col("value").cast("integer").alias("event_id"), 
        expr("CASE WHEN value %3=0 THEN 'purchase'"
             "WHEN value %3=1 THEN 'click'"
             "ELSE 'view' END").alias("event_type"),
        expr("cast(500 + (value % 100)as int)").alias("user_id"), 
        current_timestamp().alias("event_ts")
    )



#### Writing the stream


In [0]:
%sql
SHOW VOLUMES IN workspace.default;

In [0]:
query = stream_df.writeStream\
    .format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "/Volumes/workspace/default/temp/checkpoint/events")\
    .trigger(availableNow=True)\
    .toTable("events")
print("Streaming query started. Writing to main.default.events every 5 seconds")

In [0]:
# Stop the stream
query.stop()